# Multi-Task CNN Training for Out-Of-Distribution Plant Disease Identification

This notebook trains the **Multi-Task CNN** model (`train/train_multi_task_cnn.py`) according to the project specifications.

### Pipeline Overview:
1. **Google Drive Integration**: Mounts `/content/drive` to access your project folder `Nat_AML_DPD_group_assignment`.
2. **Fast Local Extraction**: Copies `DPD_Small.7z` from Drive to Colab local SSD (`/content/`) and extracts with `7z` to maximize DataLoader throughput.
3. **Multi-Task Model**: ResNet-50 (or ConvNeXt, EfficientNet, ViT) with joint crop (55) & disease (175) heads.
4. **2-Stage Protocol**: Model selection + gamma sweep on pseudo-unseen holdout validation splits, then Stage 2 refit and test evaluation.
5. **Live Google Drive Sync**: Automatically mirrors checkpoints (`last.pt`), `history.csv`, model weights, and metrics to `Nat_AML_DPD_group_assignment/runs/` during training.
6. **Resume Support**: If Colab disconnects, setting `RESUME = True` automatically restores the checkpoint from Drive.

In [1]:
# Step 1: Check GPU accelerator status
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

Tue Oct 27 18:52:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Step 2: Mount Google Drive & Verify Project Folder
from google.colab import drive
from pathlib import Path
import os

# Mount Drive
drive.mount('/content/drive')

# User project directory on Google Drive
GDRIVE_PROJECT = Path("/content/drive/MyDrive/Nat_AML_DPD_group_assignment")
GDRIVE_DATASET_7Z = GDRIVE_PROJECT / "DPD_Small.7z"
GDRIVE_RUNS = GDRIVE_PROJECT / "runs"

# Verify paths exist
assert GDRIVE_PROJECT.exists(), f"Google Drive folder not found: {GDRIVE_PROJECT}"
assert GDRIVE_DATASET_7Z.exists(), f"Dataset archive not found at: {GDRIVE_DATASET_7Z}"
GDRIVE_RUNS.mkdir(parents=True, exist_ok=True)

print("Google Drive mounted and verified successfully!")
print(f"Project folder: {GDRIVE_PROJECT}")
print(f"Dataset archive: {GDRIVE_DATASET_7Z}")
print(f"Persistent runs folder: {GDRIVE_RUNS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted and verified successfully!
Project folder: /content/drive/MyDrive/Nat_AML_DPD_group_assignment
Dataset archive: /content/drive/MyDrive/Nat_AML_DPD_group_assignment/DPD_Small.7z
Persistent runs folder: /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs


In [4]:
# Step 3: Set Repository Working Directory
import os
from pathlib import Path

# Cloned repository path on Colab
# If you have not cloned yet, you can run:
# !git clone <YOUR_GIT_REPO_URL> /content/Out-Of-Distribution-Plant-Disease-Identification

REPO_DIR = Path("/content/Out-Of-Distribution-Plant-Disease-Identification")
if not (REPO_DIR / "train").exists():
    # Fallback to current working directory if already inside repo
    REPO_DIR = Path.cwd()

os.chdir(REPO_DIR)
print(f"Current repo working directory: {Path.cwd()}")

Current repo working directory: /content


In [5]:
# Step 4: Extract Dataset to Fast Local Colab SSD (/content/)
import shutil
from pathlib import Path
import os

LOCAL_ARCHIVE = Path("/content/DPD_Small.7z")
EXTRACT_DEST = Path("/content/dataset")
EXTRACT_DEST.mkdir(parents=True, exist_ok=True)

# 1. Copy .7z from Google Drive to local SSD (if not already copied)
if not LOCAL_ARCHIVE.exists():
    print(f"Copying {GDRIVE_DATASET_7Z} to local SSD ({LOCAL_ARCHIVE})...")
    shutil.copy2(GDRIVE_DATASET_7Z, LOCAL_ARCHIVE)
    print("Copy complete!")
else:
    print(f"Local archive {LOCAL_ARCHIVE} already present.")

# 2. Extract using 7z if CSVs are not yet present
def find_dpd_root(search_dir: Path) -> Path | None:
    for candidate in [search_dir, search_dir / "DPD_Small", *search_dir.glob("**/DPD_Small*")]:
        if candidate.is_dir() and (candidate / "DPD_Small_train.csv").exists():
            return candidate
    return None

dpd_root = find_dpd_root(EXTRACT_DEST)
if dpd_root is None:
    print("Extracting dataset with 7z (fast multithreaded)...", flush=True)
    !7z x -y /content/DPD_Small.7z -o/content/dataset
    dpd_root = find_dpd_root(EXTRACT_DEST)

assert dpd_root is not None, f"Could not find DPD_Small_train.csv inside {EXTRACT_DEST}!"

# 3. Verify all dataset components exist
assert (dpd_root / "DPD_Small_train.csv").exists(), f"Missing train CSV in {dpd_root}"
assert (dpd_root / "DPD_Small_test_seen.csv").exists(), f"Missing test seen CSV in {dpd_root}"
assert (dpd_root / "DPD_Small_test_unseen.csv").exists(), f"Missing test unseen CSV in {dpd_root}"
assert (dpd_root / "DPD_Small_dataset").exists(), f"Missing DPD_Small_dataset folder in {dpd_root}"

# 4. Export DPD_ROOT environment variable (read by dpd/data.py)
os.environ["DPD_ROOT"] = str(dpd_root)
print(f"\nDataset verified successfully!")
print(f"DPD_ROOT set to: {os.environ['DPD_ROOT']}")

Copying /content/drive/MyDrive/Nat_AML_DPD_group_assignment/DPD_Small.7z to local SSD (/content/DPD_Small.7z)...
Copy complete!
Extracting dataset with 7z (fast multithreaded)...

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
  0M Scan /content                  1 file, 3399045528 bytes (3242 MiB)

Extracting archive: /content/DPD_Small.7z
--
Path = /content/DPD_Small.7z
Type = 7z
Physical Size = 3399045528
Headers Size = 107901
Method = LZMA2:25
Solid = +
Blocks = 1

      0%      1%      2%      3%      4%      5%      6%      6% 1 - DPD_Small_dataset/APPD_apple_healthy_138.jp                                                      6% 408 - DPD_Small_dataset/CC_coffee_rust_513.jp                                                    6% 564 - DPD_Small_dataset/CDC_cassava_mosaic_disease_4185.j                                                                7% 564 - DPD_Small_dataset/

In [6]:
# Step 5: Install Required Python Packages
!pip install -q pandas pillow tqdm matplotlib p7zip-full

ERROR: Could not find a version that satisfies the requirement p7zip-full (from versions: none)
ERROR: No matching distribution found for p7zip-full


In [7]:
# ================= Step 6: Experiment Parameters ================
CONFIG_PATH = "config/multitask/Nat_multitask_resnetv2_50.json"

# Continuous live sync destination (Google Drive runs folder)
SYNC_DIR = str(GDRIVE_RUNS)

# Command-line overrides applied via --set key.path=value
OVERRIDES = [
    "runtime.num_workers=2",                # Recommended for standard Colab vCPUs
    "runtime.checkpoint_every_n_epochs=5",  # Save checkpoint every 5 epochs
]

# Set RESUME=True to resume from last saved checkpoint (will auto-pull from Drive if restarted)
RESUME = False

# Quick smoke test flag (overrides to 1 epoch & 1 validation split to test pipeline in ~2 min)
SMOKE_TEST = False
if SMOKE_TEST:
    OVERRIDES.extend([
        "training.max_epochs=1",
        "validation.n_splits=1",
        "output_dir=runs/multitask/nat_multitask_v1_smoke",
    ])

print(f"Config: {CONFIG_PATH}")
print(f"Sync Dir (Google Drive): {SYNC_DIR}")
print(f"Resume: {RESUME}")
print(f"Smoke Test: {SMOKE_TEST}")
print(f"Overrides: {OVERRIDES}")

Config: config/multitask/Nat_multitask_resnetv2_50.json
Sync Dir (Google Drive): /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs
Resume: False
Smoke Test: False
Overrides: ['runtime.num_workers=2', 'runtime.checkpoint_every_n_epochs=5']


In [8]:
# Step 7: Run Multi-Task CNN Training Script
import subprocess
import sys

cmd = [
    sys.executable,
    "-m", "train.train_multi_task_cnn",
    "--config", CONFIG_PATH,
    "--sync-dir", SYNC_DIR,
]

if OVERRIDES:
    cmd.append("--set")
    cmd.extend(OVERRIDES)

if RESUME:
    cmd.append("--resume")

print("Starting training execution:")
print(" ".join(cmd))
print("=" * 70)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Training exited with error code {proc.returncode}")
print("\nTraining run finished successfully and all outputs are synced to Google Drive!")

Starting training execution:
/usr/bin/python3 -m train.train_multi_task_cnn --config config/multitask/Nat_multitask_resnetv2_50.json --sync-dir /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs --set runtime.num_workers=2 runtime.checkpoint_every_n_epochs=5
Applied overrides:
  runtime.num_workers=2
  runtime.checkpoint_every_n_epochs=5
Live synchronization enabled -> /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs/multitask/Nat_multitask_resnetv2_50

resize to 256px: 100%|██████████| 11448/11448 [01:37<00:00, 116.90it/s]

--- Validation Split 1/1 (fit 5740, val-seen 852, pseudo-unseen 559) ---
/content/train/train_multi_task_cnn.py:332: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org

In [9]:
# Step 8: Display Summary Metrics and Plot Training Curves
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

run_name = "nat_multitask_v1_smoke" if SMOKE_TEST else "nat_multitask_v1"
# Check both local run directory and Google Drive synced run directory
local_out_dir = Path("runs/multitask") / run_name
gdrive_out_dir = GDRIVE_RUNS / "multitask" / run_name
out_dir = gdrive_out_dir if gdrive_out_dir.exists() else local_out_dir

metrics_file = out_dir / "metrics.json"
history_file = out_dir / "history.csv"

if metrics_file.exists():
    with open(metrics_file, "r") as f:
        m = json.load(f)
    print("================ Evaluation Summary ================")
    print(f"Version: {m.get('version')}")
    print(f"Selected Epoch: {m.get('selected_epoch')}, Calibrated Gamma: {m.get('gamma')}")
    print(f"Harmonic Mean Top-1: {m.get('harmonic_mean_top1', 0):.2%}")
    print(f"Harmonic Mean Top-5: {m.get('harmonic_mean_top5', 0):.2%}")
    print(f"Seen Top-1 Accuracy: {m.get('seen', {}).get('top1_accuracy', 0):.2%}")
    print(f"Unseen Top-1 Accuracy: {m.get('unseen', {}).get('top1_accuracy', 0):.2%}")
    print(f"Uncalibrated HM (gamma=0): {m.get('harmonic_mean_top1_gamma0', 0):.2%}")
    print(f"Results location: {out_dir}")

if history_file.exists():
    df_hist = pd.read_csv(history_file)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curve
    for stage, group in df_hist.groupby("stage"):
        axes[0].plot(group["epoch"], group["loss"], marker="o", label=stage)
    axes[0].set_title("Training Loss by Stage")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True)
    axes[0].legend()

    # Validation HM curve
    val_splits = df_hist[df_hist["stage"].str.startswith("split")]
    if not val_splits.empty and "val_hm_best" in val_splits.columns:
        for stage, group in val_splits.groupby("stage"):
            axes[1].plot(group["epoch"], group["val_hm_best"], marker="s", label=stage)
        axes[1].set_title("Validation Harmonic Mean (Best Gamma)")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("HM Top-1")
        axes[1].grid(True)
        axes[1].legend()

    plt.tight_layout()
    plt.show()